<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit0/w03-classes-and-contracts/notebook.ipynb)


# Unit 3 — Classes and contracts

**Week 0 · prerequisite · about 45 minutes**

**Goal:** Enough dataclass to read the typed contracts this course is built on, and the two bugs that bite hardest.

**Why it matters:** The course passes typed objects across every boundary: `Document`, `Tool`, `ResearchAnswer`. Two dataclass mistakes account for most of the confusing bugs people hit with them, and both are visible in ten lines.

Some cells below ship **broken on purpose**, marked `<------ EDIT THIS LINE`. Run them first and
read what happens. Debugging something wrong teaches more than filling in a blank.

In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [2]:
from bootcamp_agent.checks import check, review
from bootcamp_agent.hints import hint  # noqa: F401 - hint("w01-e2") when you want a nudge
import bootcamp_agent.week0_checks  # noqa: F401 — importing is what registers them

## 1. The mutable default

**Context.** The stub below runs. Run it and read both lines before changing anything — the
second call is wrong, and the reason is one of the most famous surprises in Python.

**Instructions.**

1. Run it first. `second call` should print `['beta']`, and it will not.
2. Fix the default so each call gets its own list.
3. The rule: a default argument is evaluated **once**, when the function is defined.

In [7]:
def add_tag(tag, tags=None):   # <------ EDIT THIS LINE
    if tags == None:
        tags = []
    tags.append(tag)
    return tags


print(f"first  call -> {add_tag('alpha')}")
print(f"second call -> {add_tag('beta')}   <- should be ['beta']")

first  call -> ['alpha']
second call -> ['beta']   <- should be ['beta']


**Expected output**

```
first  call -> ['alpha']
second call -> ['beta']   <- should be ['beta']
✅ w03-e1 passed
```

Before the fix the second call returns `['alpha', 'beta']`: one list, shared by every call for
the life of the process.

In [8]:
check("w03-e1", add_tag)

✅ w03-e1 passed


True

## 2. Frozen, and why a tool definition must be

**Context.** `Tool` in this package is a frozen dataclass. Anything holding a reference to a
tool can read it and cannot change it — which matters when the thing holding the reference is
a model's output path.

**Instructions.** Make `ToolDef` frozen, so reassigning `name` raises.

In [23]:
from dataclasses import dataclass


@dataclass(frozen=True)
class ToolDef:            # <------ EDIT THIS LINE
    name: str
    description: str


probe = ToolDef(name="search", description="find things")
# The try/except is the point of the exercise, not scaffolding: once the
# class is frozen the assignment RAISES, and an exercise whose correct
# answer crashes the notebook cannot show you that it worked.
try:
    probe.name = "anything at all"
    print(f"name is now {probe.name!r}  <- it should not have been possible to do that")
except Exception as error:
    print(f"refused, as it should: {type(error).__name__}")


FrozenInstanceError: cannot assign to field 'name'

**Expected output**

```
refused, as it should: FrozenInstanceError
✅ w03-e2 passed
```

In [11]:
check("w03-e2", ToolDef)

✅ w03-e2 passed


True

## 3. Why the answer type fails closed

**Context.** `parse_research_answer` turns a model's JSON into a `ResearchAnswer`. It refuses a
payload that carries a field the schema does not declare.

**Instructions.** Try it below, then answer. Think about what an accepted unknown field would
let a model do.

In [32]:
from bootcamp_agent.schema import parse_research_answer
import json

payload = json.dumps({
    "answer": "Chunking splits a document.", "citations": ["rag-basics"],
    "confidence": 0.8, "needs_human_review": False,
    "confidence_override": 1.0,          # a field the schema never declared
})
try:
    print(parse_research_answer(payload))
except Exception as error:
    print(f"{type(error).__name__}: {error}")

verdict = {
    "rejects_unknown_fields": True,   # <------ EDIT: you just ran it. What happened?
    "why_fail_closed": (
        "A model that can add fields can add one the code later reads by accident. Refusing "
        "the whole payload means a made-up field is a loud failure now, not a quiet one later."
    ),
}

AnswerParseError: Wrong fields: missing=[] unknown=['confidence_override']


**Expected output**

```
AnswerParseError: unknown field 'confidence_override'
✅ w03-e3 passed
```

In [33]:
check("w03-e3", verdict)

✅ w03-e3 passed


True

## Review

The scorecard for this unit. Every ❌ line names the exercise and the hint.

In [34]:
review("w03")

w03: 3/3 passed  ·  300/300 marks


True